In [12]:
import os
import gc
import sys
import glob
import numpy as np
import pandas as pd
import netCDF4 as nc
from datetime import datetime, timedelta
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from tqdm import tqdm  # Progress bar library
import multiprocessing as mp

In [13]:
# To use PLUMBER2_GPP_common_utils, change directory to where it exists
os.chdir('/srv/ccrc/LandAP/z5218916/script/PLUMBER2/LSM_GPP_PLUMBER2')
from PLUMBER2_GPP_common_utils import *

In [14]:
models_calc_LAI   = ['ORC2_r6593','ORC2_r6593_CO2','ORC3_r7245_NEE','ORC3_r8120','GFDL','SDGVM','QUINCY','NoahMPv401']
model_LAI_names   = {'ORC2_r6593':'lai','ORC2_r6593_CO2':'lai','ORC3_r7245_NEE':'lai','ORC3_r8120':'lai',
                     'GFDL':'lai', 'SDGVM':'lai','QUINCY':'LAI','NoahMPv401':'LAI'} #

### Calculate random forest

In [4]:
def get_annual_mean(model_in, site_name, var_name):
    
    secondly_to_annually = 3600*24*365.
    
    models_calc_LAI    = ['ORC2_r6593','ORC2_r6593_CO2','ORC3_r7245_NEE','ORC3_r8120','GFDL','SDGVM','QUINCY','NoahMPv401']
    PLUMBER2_path_site = f"/srv/ccrc/LandAP/z5218916/script/PLUMBER2/LSM_GPP_PLUMBER2/nc_files/{site_name}.nc"
    PLUMBER2_met_path  = "/srv/ccrc/LandAP/z5218916/data/Fluxnet_data/Post-processed_PLUMBER2_outputs/Nc_files/Met/"
    file_met_path      = glob.glob(PLUMBER2_met_path+"/*"+site_name+"*.nc")
    
    # prepare dataset
    f                  = nc.Dataset(PLUMBER2_path_site, mode='r')
    f_met              = nc.Dataset(file_met_path[0], mode='r')

    var_in             = pd.DataFrame(f.variables['obs_Tair'][:].data-273.15, columns=['Tair'])
    var_in['SWdown']   = f.variables['obs_SWdown'][:].data

    var_in['CO2']      = f_met.variables['CO2air'][:,0,0].data
    var_in['VPD']      = f_met.variables['VPD'][:,0,0].data
    var_in['Precip']   = f_met.variables['Precip'][:,0,0].data

    # Read time
    time   = nc.num2date(f.variables['CABLE_time'][:],f.variables['CABLE_time'].units,
                         only_use_cftime_datetimes=False,only_use_python_datetimes=True)
    ntime  = len(time)
    year  = np.zeros(ntime)

    for tt,t in enumerate(time):
        year[tt] = t.year

    var_in['year'] = year
    
    if var_name == 'NEE' and (model_in == 'NoahMPv401' or model_in == 'GFDL' or model_in == 'STEMMUS-SCOPE'):
        var_in[var_name] = f.variables[f'{model_in}_{var_name}'][:].data*(-1)
    else:
        var_in[var_name] = f.variables[f'{model_in}_{var_name}'][:].data

    if model_in in models_calc_LAI:
        var_in['LAI'] = read_LAI_model(site_name, model_in, model_LAI_names[model_in])
    else:
        var_in['LAI'] = read_LAI_obs(site_name, PLUMBER2_met_path)

    try: 
        var_in['SMtop1m'] = f.variables[f'{model_in}_SMtop1m'][:].data
    except: 
        var_in['SMtop1m'] = f.variables['model_mean_SMtop1m'][:].data
        
    # Change units from gC/m2/s to gC/m2/year
    if var_name in ['NEE','GPP']:
        var_in.loc[:,var_name] = var_in[var_name][:]*secondly_to_annually
    
    var_mean  = var_in.groupby(['year']).mean(numeric_only=True)

    return var_mean

In [5]:
def save_annual_mean(var_name, model_in):

    remove_site        = get_removed_site_names()
    site_names, IGBP_types, clim_types, model_names = load_default_list()

    PLUMBER2_met_path  = "/srv/ccrc/LandAP/z5218916/data/Fluxnet_data/Post-processed_PLUMBER2_outputs/Nc_files/Met/"
    sites_IGBP         = read_IGBP_veg_type(site_names, PLUMBER2_met_path)

    # Initialize an empty list to hold the dataframes
    all_var_means = []

    # Loop over site names
    for site_name in site_names:
        if site_name not in remove_site:
            try:
                var_mean         = get_annual_mean(model_in, site_name, var_name)
                var_mean['IGBP'] = sites_IGBP[site_name]
                all_var_means.append(var_mean)
            except Exception as e:
                print(f'Error occurred while processing {model_in} at {site_name}: {str(e)}')                
                
    # Concatenate all the var_mean dataframes
    var_means = pd.concat(all_var_means, ignore_index=True)
    var_means.to_csv(f'./txt/{model_in}_for_RF_annual_{var_name}.csv')


### Run random forest

<h4 style="color:green;"> Simple random forest </h4>

In [5]:
def calc_random_forest_simple():

    var_in = pd.read_csv('./txt/data_for_RF_annual_all_sites.csv')
    var_in_cleaned = var_in.dropna()
    
    # Define the features (X) and the target (y)
    X = var_in_cleaned[['Tair', 'SWdown', 'CO2', 'VPD', 'Precip', 'LAI', 'SMtop1m']]  # Predictors
    y = var_in_cleaned['NEE']  # Target variable

    # Split the data into training (80%) and testing (20%) sets
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    # Initialize the Random Forest Regressor
    rf_model = RandomForestRegressor(n_estimators=100, random_state=42, warm_start=True)

    # Train the model with tqdm to track progress
    n_trees = 100  # Number of trees in the random forest
    for i in tqdm(range(1, n_trees + 1)):
        rf_model.set_params(n_estimators=i)  # Increase the number of trees progressively
        rf_model.fit(X_train, y_train)  # Fit the model for each iteration

    # Make predictions on the test set
    y_pred = rf_model.predict(X_test)

    # Evaluate the model
    mse = mean_squared_error(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    print(f"Mean Squared Error (MSE): {mse}")
    print(f"R-squared (R2 Score): {r2}")

    # Optional: Display feature importance
    feature_importance = pd.Series(rf_model.feature_importances_, index=X.columns)
    print("Feature Importances:")
    print(feature_importance.sort_values(ascending=False))
    var_in = None
    var_in_cleaned = None

<h4 style="color:green;"> Complicate random forest </h4>

<h5 style="color:orange;"> Using IGBP as one-hot</h5>

In [6]:
def calc_random_forest_with_one_hot(var_name, model_in, n_trees=100):
    
    var_in = pd.read_csv(f'./txt/{model_in}_for_RF_annual_{var_name}.csv')
    var_in_cleaned = var_in.dropna()

    # Define the features (X) and the target (y)
    X = var_in_cleaned[['Tair', 'SWdown', 'CO2', 'VPD', 'Precip', 'LAI', 'SMtop1m', 'IGBP']]  # Predictors
    y = var_in_cleaned['NEE']  # Target variable

    # One-hot encode the 'IGBP' column
    igbp_one_hot = pd.get_dummies(X['IGBP'], prefix='IGBP', drop_first=True)
    print('igbp_one_hot',igbp_one_hot)
    
    # Combine the one-hot encoded IGBP with the existing predictors
    X = pd.concat([X.drop(columns=['IGBP']), igbp_one_hot], axis=1)

    # Standardize the predictors
    scaler_X = StandardScaler()
    X_normalized = scaler_X.fit_transform(X)

    # Standardize the target variable
    scaler_y = StandardScaler()
    y_normalized = scaler_y.fit_transform(y.values.reshape(-1, 1)).flatten()

    # Split the data into training (80%) and testing (20%) sets
    X_train, X_test, y_train, y_test = train_test_split(X_normalized, y_normalized, test_size=0.2, random_state=42)

    # Initialize the Random Forest Regressor
    rf_model = RandomForestRegressor(n_estimators=n_trees, random_state=42, warm_start=True)

    # Train the model with tqdm to track progress
    for i in tqdm(range(1, n_trees + 1)):
        rf_model.set_params(n_estimators=i)  # Increase the number of trees progressively
        rf_model.fit(X_train, y_train)  # Fit the model for each iteration

    # Make predictions on the test set
    y_pred_standardized = rf_model.predict(X_test)
    
    # Destandardize the predictions
    y_pred = scaler_y.inverse_transform(y_pred_standardized.reshape(-1, 1)).flatten()

    # Evaluate the model using destandardized predictions
    mse = mean_squared_error(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    rmse = np.sqrt(mse)  # Calculate RMSE
    print(f"Mean Squared Error (MSE): {mse}")
    print(f"Mean Absolute Error (MAE): {mae}")
    print(f"R-squared (R2 Score): {r2}")
    print(f"Root Mean Squared Error (RMSE): {rmse}")

    # Perform cross-validation
    cv = KFold(n_splits=5, shuffle=True, random_state=42)  # 5-fold cross-validation

    # Cross-validated MSE
    cv_scores_mse = cross_val_score(rf_model, X_normalized, y_normalized, cv=cv, scoring='neg_mean_squared_error')
    cv_mse = -np.mean(cv_scores_mse)  # Convert negative MSE to positive
    cv_rmse = np.sqrt(cv_mse)  # Calculate RMSE from CV MSE
    print(f"Cross-Validated Mean Squared Error (MSE): {cv_mse}")
    print(f"Cross-Validated Root Mean Squared Error (RMSE): {cv_rmse}")

    # Cross-validated R²
    cv_scores_r2 = cross_val_score(rf_model, X_normalized, y_normalized, cv=cv, scoring='r2')
    cv_r2 = np.mean(cv_scores_r2)  # Calculate the mean R² score across folds
    print(f"Cross-Validated R-squared (R2 Score): {cv_r2}")

    # Optional: Display feature importance
    feature_importance = pd.Series(rf_model.feature_importances_, index=X.columns)
    print("Feature Importances:")
    print(feature_importance.sort_values(ascending=False))

    # Clean up
    var_in = None
    var_in_cleaned = None


<h5 style="color:orange;"> Using IGBP as numerical value</h5>

In [15]:
def calc_random_forest(var_name, model_in, n_trees=100):
    
    from scipy.stats import pearsonr  # Import the correlation function
    var_in = pd.read_csv(f'./txt/{var_name}_annual_value/{var_name}_annual_value_{model_in}.csv')
    var_in_cleaned = var_in.dropna()
    

    # Define the features (X) and the target (y)
    X = var_in_cleaned[['Tair', 'SWdown', 'VPD', 'LAI', 'SMtop1m','Precip', 'IGBP', 'CO2', ]]  # Predictors 
    y = var_in_cleaned['NEE'] # Target variable

    # Label encode the 'IGBP' column
    label_encoder    = LabelEncoder()
    X.loc[:, 'IGBP'] = label_encoder.fit_transform(X['IGBP']) 
    
    
    # Standardize the predictors
    scaler_X = StandardScaler()
    X_normalized = scaler_X.fit_transform(X)

    # # Standardize the target variable
    # scaler_y = StandardScaler()
    # y_normalized = scaler_y.fit_transform(y.values.reshape(-1, 1)).flatten()
    y_normalized = y
    
    # Split the data into training (80%) and testing (20%) sets
    X_train, X_test, y_train, y_test = train_test_split(X_normalized, y_normalized, test_size=0.2, random_state=42)

    
    mse           = np.zeros(len(random_states))
    mae           = np.zeros(len(random_states))
    cor           = np.zeros(len(random_states))
    r2            = np.zeros(len(random_states))
    rmse          = np.zeros(len(random_states))
    
    cv_scores_mse = np.zeros(len(random_states))
    cv_mse        = np.zeros(len(random_states))
    cv_rmse       = np.zeros(len(random_states))
    cv_r2         = np.zeros(len(random_states))
    
    feature_importance = {}
    
    # Initialize the Random Forest Regressor
    # !!! Try to use different seeds random_states
    random_states = np.arange(0,100)
    
    for r, random_state in enumerate(random_states):
        
        # for random_state in random_states:    
        rf_model = RandomForestRegressor(n_estimators=n_trees, random_state=random_state, warm_start=True)

        # Train the model with tqdm to track progress
        for i in tqdm(range(1, n_trees + 1)):
            rf_model.set_params(n_estimators=i)  # Increase the number of trees progressively
            rf_model.fit(X_train, y_train)  # Fit the model for each iteration

        # Make predictions on the test set
        y_pred_standardized = rf_model.predict(X_test)
        y_pred = y_pred_standardized
        # # Destandardize the predictions
        # y_pred = scaler_y.inverse_transform(y_pred_standardized.reshape(-1, 1)).flatten()

        # Evaluate the model using destandardized predictions
        mse[r]    = mean_squared_error(y_test, y_pred)
        mae[r]    = mean_absolute_error(y_test, y_pred)
        cor[r], _ = pearsonr(y_test, y_pred)
        r2[r]     = r2_score(y_test, y_pred)
        rmse[r]   = np.sqrt(mse)  # Calculate RMSE
        
        # Perform cross-validation
        cv = KFold(n_splits=5, shuffle=True, random_state=random_state)  # 5-fold cross-validation

        # Cross-validated MSE
        cv_scores_mse[r] = cross_val_score(rf_model, X_normalized, y_normalized, cv=cv, scoring='neg_mean_squared_error')
        cv_mse[r]        = -np.mean(cv_scores_mse)  # Convert negative MSE to positive
        cv_rmse[r]       = np.sqrt(cv_mse)  # Calculate RMSE from CV MSE
        # Cross-validated R²
        cv_scores_r2 = cross_val_score(rf_model, X_normalized, y_normalized, cv=cv, scoring='r2')
        cv_r2[r] = np.mean(cv_scores_r2)  # Calculate the mean R² score across folds


        # Optional: Display feature importance
        feature_importance[r] = pd.Series(rf_model.feature_importances_, index=X.columns)


    print(f"Mean Squared Error (MSE): {np.mean(mse)}")
    print(f"Mean Absolute Error (MAE): {np.mean(mae)}")
    print(f"cor (cor): {np.mean(cor)}")
    print(f"R-squared (R2 Score): {np.mean(r2)}")
    print(f"Root Mean Squared Error (RMSE): {np.mean(rmse)}")
    print(f"Cross-Validated Mean Squared Error (MSE): {np.mean(cv_mse)}")
    print(f"Cross-Validated Root Mean Squared Error (RMSE): {np.mean(cv_rmse)}")
    print(f"Cross-Validated R-squared (R2 Score): {np.mean(cv_r2)}")
    
    print("Feature Importances:")
    print(feature_importance)
    # print(feature_importance.sort_values(ascending=False))
    
    # Clean up
    var_in = None
    var_in_cleaned = None


In [11]:
var_name           = 'NEE'

# PLUMBER2_path_site = "/srv/ccrc/LandAP/z5218916/script/PLUMBER2/LSM_GPP_PLUMBER2/nc_files/AU-How.nc"
# f                  = nc.Dataset(PLUMBER2_path_site, mode='r')
# model_list         = f.variables[f'{var_name}_models'][:]
# model_list         = model_list.tolist()
# model_list.append('obs')
# f.close()

model_list = [ 'obs'] #, 'CABLE', 'STEMMUS-SCOPE', 'CLM5'
for model_in in model_list:
    print('Model is ', model_in)
    # save_annual_mean(var_name, model_in)
    # calc_random_forest(var_name, model_in, n_trees=50)
    # calc_random_forest(var_name, model_in, n_trees=100)
    calc_random_forest(var_name, model_in, n_trees=200)

Model is  obs


KeyError: "['LAI'] not in index"

In [ ]:
var_name   = 'NEE'
model_list = [ 'obs', 'QUINCY', 'STEMMUS-SCOPE', 'ORC2_r6593', 'ORC3_r8120']
for model_in in model_list:
    print('Model is ', model_in)
    save_annual_mean(var_name, model_in)
    calc_random_forest(var_name, model_in, n_trees=50)
    calc_random_forest(var_name, model_in, n_trees=100)
    calc_random_forest(var_name, model_in, n_trees=200)

Model is  obs


100%|██████████| 50/50 [00:00<00:00, 107.26it/s]


Mean Squared Error (MSE): 87113.4340080261
Mean Absolute Error (MAE): 186.35146290690267
R-squared (R2 Score): 0.4767305722714841
Root Mean Squared Error (RMSE): 295.149850089791
Cross-Validated Mean Squared Error (MSE): 136235.43284651433
Cross-Validated Root Mean Squared Error (RMSE): 369.10084373584726
Cross-Validated R-squared (R2 Score): 0.4571203506125222
Feature Importances:
SWdown     0.359838
LAI        0.176271
Tair       0.163268
SMtop1m    0.081886
IGBP       0.065540
VPD        0.063918
CO2        0.049704
Precip     0.039575
dtype: float64


100%|██████████| 100/100 [00:00<00:00, 127.59it/s]


Mean Squared Error (MSE): 86707.45256501969
Mean Absolute Error (MAE): 184.19169797198225
R-squared (R2 Score): 0.47916920506985117
Root Mean Squared Error (RMSE): 294.46129213365157
Cross-Validated Mean Squared Error (MSE): 131455.5202725898
Cross-Validated Root Mean Squared Error (RMSE): 362.5679526276279
Cross-Validated R-squared (R2 Score): 0.4864423146493101
Feature Importances:
SWdown     0.363054
LAI        0.186494
Tair       0.150409
SMtop1m    0.081437
IGBP       0.066427
VPD        0.061177
CO2        0.050535
Precip     0.040467
dtype: float64


...

Mean Squared Error (MSE): 42728.55825508111
Mean Absolute Error (MAE): 130.5643680980354
R-squared (R2 Score): 0.9296886402260556
Root Mean Squared Error (RMSE): 206.70887318903635
Cross-Validated Mean Squared Error (MSE): 48699.0904788027
Cross-Validated Root Mean Squared Error (RMSE): 220.67870418054093
Cross-Validated R-squared (R2 Score): 0.9255544361680809
Feature Importances:
Precip     0.389892
LAI        0.374896
SMtop1m    0.076170
SWdown     0.072664
Tair       0.041706
VPD        0.026166
CO2        0.010793
IGBP       0.007712
dtype: float64


100%|██████████| 100/100 [00:00<00:00, 135.51it/s]


Mean Squared Error (MSE): 43029.59805248027
Mean Absolute Error (MAE): 129.33469306357964
R-squared (R2 Score): 0.929193268550399
Root Mean Squared Error (RMSE): 207.43576849829992
Cross-Validated Mean Squared Error (MSE): 47199.69231558179
Cross-Validated Root Mean Squared Error (RMSE): 217.25490170668598
Cross-Validated R-squared (R2 Score): 0.9277566155068264
Feature Importances:
LAI        0.391611
Precip     0.377113
SMtop1m    0.073673
SWdown     0.071781
Tair       0.043843
VPD        0.025048
CO2        0.009423
IGBP       0.007508
dtype: float64


100%|██████████| 200/200 [00:01<00:00, 134.21it/s]


Mean Squared Error (MSE): 40978.165519979295
Mean Absolute Error (MAE): 127.55056304326537
R-squared (R2 Score): 0.9325689736229542
Root Mean Squared Error (RMSE): 202.43064372762166
Cross-Validated Mean Squared Error (MSE): 46943.324327226845
Cross-Validated Root Mean Squared Error (RMSE): 216.66408176536055
